In [1]:
import os
import scanpy as sc
import anndata as ad
import muon as mu
import pandas as pd
import glob
import gc
import warnings
import snapatac2 as sa
import subprocess

/home/hwjeong/miniconda3/envs/project_md/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
# Suppress downsteam processing warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# =========================================================================
# 1. LOAD RNA DATA & CREATE SAMPLE-AWARE CELL BARCODE WHITELISTS
# =========================================================================
# Load the preprocessed scRNA-seq MuData object
mdata = mu.read("data/rna.h5mu")
rna_obs_names = set(mdata.mod["rna"].obs_names)

# Split the verified cell barcodes by their respective sample IDs
rna_by_sample = {}
for bc in mdata.mod["rna"].obs_names:
    sample = bc.split("-")[-1]  # Extract sample suffix (e.g., 'VM1')
    core_bc = bc.split("-")[0] + "-1"  # Extract raw 10X barcode format
    rna_by_sample.setdefault(sample, set()).add(core_bc)

# =========================================================================
# 2. FILE SEARCH & BEDTOOLS-BASED CONSENSUS PEAK GENERATION
# =========================================================================
# Locate all individual sample peak files
atac_dir = "data/atac"
out_dir = "data/atac_processed"
os.makedirs(out_dir, exist_ok=True)
individual_peak_files = sorted(glob.glob(os.path.join(atac_dir, "*_atac_peaks.bed.gz")))

print("=== Phase 1: Generating Global Consensus Peak Set via Bedtools ===")
consensus_peak_file = os.path.join(out_dir, "global_consensus_peaks.bed")

# Merge overlapping genomic intervals across all samples into unified consensus peaks
try:
    shell_command = (
        f"zcat {os.path.join(atac_dir, '*_atac_peaks.bed.gz')} | "
        f"grep -v '^#' | "
        f"sort -k1,1 -k2,2n | "
        f"bedtools merge -i - > {consensus_peak_file}"
    )
    subprocess.run(shell_command, shell=True, check=True)
    print(f"Consensus peak file successfully created at: {consensus_peak_file}")
except Exception as e:
    raise RuntimeError(f"Bedtools execution failed! Error: {e}")

# =========================================================================
# 3. MEMORY-SAFE SEQUENTIAL ATAC COUNT MATRIX GENERATION
# =========================================================================
processed_h5ads = []
print("\n=== Phase 2: Processing Samples with Consensus Matrix Quantification ===")

for peak_path in individual_peak_files:
    fname = os.path.basename(peak_path)
    parts = fname.split("_")
    
    gsm_id = parts[0]       
    sample_id = parts[1]    
    group_id = "".join([char for char in sample_id if char.isalpha()]) 

    print(f"Processing sample: {sample_id} ...", end=" ", flush=True)
    
    # Safely skip samples that have no corresponding cells in the RNA modality
    sample_whitelist = list(rna_by_sample.get(sample_id, []))
    if not sample_whitelist:
        print("SKIPPED! (No matching cells found in RNA modality due to prior QC filtering)")
        continue

    frag_path = peak_path.replace("_atac_peaks.bed.gz", "_atac_fragments.tsv.gz")
    temp_file = os.path.join(out_dir, f"tmp_{sample_id}.h5ad")
    out_file = os.path.join(out_dir, f"atac_{sample_id}.h5ad")

    try:
        # 3A. Import raw fragment entries using the sample-specific verified whitelist
        adata = sa.pp.import_fragments(
            frag_path,
            chrom_sizes=sa.genome.hg38,
            sorted_by_barcode=False,
            whitelist=sample_whitelist,
            file=temp_file
        )
        
        # 3B. Construct the count matrix using the unified global consensus peak file
        adata = sa.pp.make_peak_matrix(
            adata,
            peak_file=consensus_peak_file,
            counting_strategy="fragment"
        )

        # 3C. Inject metadata layers
        adata.obs["sample_id"] = sample_id
        adata.obs["gsm_id"] = gsm_id
        adata.obs["group_id"] = group_id

        # Standardize cell names to match the scRNA-seq naming format
        adata.obs_names = [f"{bc}-{sample_id}" for bc in adata.obs_names]

        # 3D. Strict matching with final scRNA-seq high-quality barcodes
        adata = adata[adata.obs_names.isin(rna_obs_names)].copy()

        if adata.n_obs == 0:
            print("SKIPPED! (Zero cells retained after cross-modal matching)")
            if os.path.exists(temp_file): os.remove(temp_file)
            continue

        # 3E. Push data slice onto the hard drive
        adata.write_h5ad(out_file)
        processed_h5ads.append(out_file)
        print(f"Saved! (Cells: {adata.n_obs})")

    except Exception as e:
        print(f"FAILED on sample {sample_id}: {e}")

    finally:
        if 'adata' in locals(): del adata
        if os.path.exists(temp_file): os.remove(temp_file)
        gc.collect()

=== Phase 1: Generating Global Consensus Peak Set via Bedtools ===
Consensus peak file successfully created at: data/atac_processed/global_consensus_peaks.bed

=== Phase 2: Processing Samples with Consensus Matrix Quantification ===
Processing sample: VM1 ... 

... storing 'sample_id' as categorical
... storing 'gsm_id' as categorical
... storing 'group_id' as categorical


Saved! (Cells: 1733)
Processing sample: VM2 ... 

... storing 'sample_id' as categorical
... storing 'gsm_id' as categorical
... storing 'group_id' as categorical


Saved! (Cells: 1603)
Processing sample: VM3 ... 

...

Saved! (Cells: 3068)
Processing sample: MI23 ... 

... storing 'sample_id' as categorical
... storing 'gsm_id' as categorical
... storing 'group_id' as categorical


Saved! (Cells: 12)
Processing sample: MD24 ... 

... storing 'sample_id' as categorical
... storing 'gsm_id' as categorical
... storing 'group_id' as categorical


Saved! (Cells: 76)


Following standard RNA-level QC, samples **MI23** and **MD24** retained a substantial number of cells. However, a severe drop in cell counts was observed after cross-modal matching with the scATAC-seq fragment data. This indicates a high level of sequencing dropout unique to the ATAC libraries, meaning only cells with high-quality data in both modalities were kept for downstream integration.

| Sample ID | Post-RNA QC Cells | Final Multi-omic Cells | Filtered Count |
| :--- | :---: | :---: | :---: |
| **MI23** | 487 | 12 | -475 |
| **MD24** | 2,297 | 76 | -2,221 |
| **HC20** | 0 | 0 | 0 |

In [9]:
# =========================================================================
# 4. STREAMLINED COMBINATION AND CONCATENATION
# =========================================================================
print("\n=== Phase 3: Merging Standardized Samples via Disk Automation ===")
final_atac_h5ad = os.path.join(out_dir, "combined_atac_consensus_final.h5ad")

# Concatenate sample matrices on disk using the global consensus peaks
ad.experimental.concat_on_disk(
    in_files=processed_h5ads,
    out_file=final_atac_h5ad,
    join="inner"
)

# Load the consolidated master ATAC matrix into memory
combined_atac = ad.read_h5ad(final_atac_h5ad)
combined_atac.var_names_make_unique()

print("Combined ATAC Matrix Summary:")
print(combined_atac)

# =========================================================================
# 5. PACKAGING THE UNIFIED MULTI-OMIC H5MU DISK FILE
# =========================================================================
print("\n=== Phase 4: Constructing the Final MuData Object ===")

atac_dir = "data/atac"

# Registry map anchoring each sample_id to its absolute/relative raw fragment file
fragment_registry = {
    "VM1":  os.path.join(atac_dir, "GSM8306531_VM1_atac_fragments.tsv.gz"),
    "VM2":  os.path.join(atac_dir, "GSM8306533_VM2_atac_fragments.tsv.gz"),
    "VM3":  os.path.join(atac_dir, "GSM8306544_VM3_atac_fragments.tsv.gz"),
    "VM4":  os.path.join(atac_dir, "GSM8306545_VM4_atac_fragments.tsv.gz"),
    "HC5":  os.path.join(atac_dir, "GSM8306536_HC5_atac_fragments.tsv.gz"),
    "HC6":  os.path.join(atac_dir, "GSM8306537_HC6_atac_fragments.tsv.gz"),
    "MI7":  os.path.join(atac_dir, "GSM8306538_MI7_atac_fragments.tsv.gz"),
    "MI8":  os.path.join(atac_dir, "GSM8306539_MI8_atac_fragments.tsv.gz"),
    "MD9":  os.path.join(atac_dir, "GSM8306541_MD9_atac_fragments.tsv.gz"),
    "MD10": os.path.join(atac_dir, "GSM8306543_MD10_atac_fragments.tsv.gz"),
    "MD11": os.path.join(atac_dir, "GSM8306544_MD11_atac_fragments.tsv.gz"),
    "MD12": os.path.join(atac_dir, "GSM8306545_MD12_atac_fragments.tsv.gz"),
    "VM13": os.path.join(atac_dir, "GSM8306546_VM13_atac_fragments.tsv.gz"),
    "MI14": os.path.join(atac_dir, "GSM8306547_MI14_atac_fragments.tsv.gz"),
    "MD15": os.path.join(atac_dir, "GSM8306548_MD15_atac_fragments.tsv.gz"),
    "MI16": os.path.join(atac_dir, "GSM8306549_MI16_atac_fragments.tsv.gz"),
    "MD17": os.path.join(atac_dir, "GSM8306550_MD17_atac_fragments.tsv.gz"),
    "MD18": os.path.join(atac_dir, "GSM8306551_MD18_atac_fragments.tsv.gz"),
    "HC19": os.path.join(atac_dir, "GSM8306552_HC19_atac_fragments.tsv.gz"),
    "HC21": os.path.join(atac_dir, "GSM8306554_HC21_atac_fragments.tsv.gz"),
    "HC22": os.path.join(atac_dir, "GSM8306555_HC22_atac_fragments.tsv.gz"),
    "MI23": os.path.join(atac_dir, "GSM8306556_MI23_atac_fragments.tsv.gz"),
    "MD24": os.path.join(atac_dir, "GSM8306557_MD24_atac_fragments.tsv.gz")
}

# Explicitly map and attach the raw fragment filepath metadata to every individual cell barcode
combined_atac.obs["fragment_file"] = (
    combined_atac.obs["sample_id"]
    .map(fragment_registry)
    .astype(str)
)

# Store sanitized HDF5-compliant backup of the registry within the object metadata space
combined_atac.uns["fragment_registry"] = {
    str(k): str(v) for k, v in fragment_registry.items()
}

# Bind the file registry to standard slots to enable automated recognition by Muon and SnapATAC2 QC tools
if "files" not in combined_atac.uns:
    combined_atac.uns["files"] = {}
combined_atac.uns["files"]["fragments"] = combined_atac.uns["fragment_registry"]

# -------------------------------------------------------------------------
# Insert the highly optimized ATAC modality into the master MuData framework
# -------------------------------------------------------------------------
mdata.mod["atac"] = combined_atac

# Harmonize RNA local metadata datatype prior to update
if "rna" in mdata.mod and "mt" in mdata.mod["rna"].var.columns:
    mdata.mod["rna"].var["mt"] = mdata.mod["rna"].var["mt"].astype(bool)

# Synchronize multidimensional indices across multi-omic modalities
mdata.update()

# Enforce standard boolean datatype for the global master feature table
if "mt" in mdata.var.columns:
    mdata.var["mt"] = mdata.var["mt"].astype(bool)

# -------------------------------------------------------------------------
# Convert all remaining object columns to clean string structures
# -------------------------------------------------------------------------
# Purge mixed object types from global MuData arrays
for df in [mdata.var, mdata.obs]:
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str)

# Purge mixed object types from local modality arrays
for mod in mdata.mod.values():
    for df in [mod.obs, mod.var]:
        for col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].astype(str)

# -------------------------------------------------------------------------
# Finalize memory layout and write configuration
# -------------------------------------------------------------------------
mdata = mdata.copy()

final_output_path = os.path.abspath(os.path.join(out_dir, "..", "multiome.h5mu"))
print(f"Writing fully integrated multi-omic package to: {final_output_path}")

# Execute disk serialization with complete structural enforcement
mdata.write(final_output_path)

# -------------------------------------------------------------------------
# Cleanup temporary files
# -------------------------------------------------------------------------
if os.path.exists(consensus_peak_file):
    os.remove(consensus_peak_file)

if os.path.exists(final_atac_h5ad):
    os.remove(final_atac_h5ad)

for path in processed_h5ads:
    if os.path.exists(path):
        os.remove(path)

print("\n=== SUCCESS: INPUT PREPROCESSING COMPLETE ===")
print(mdata)



=== Phase 3: Merging Standardized Samples via Disk Automation ===
Combined ATAC Matrix Summary:
AnnData object with n_obs × n_vars = 39684 × 124119
    obs: 'n_fragment', 'frac_dup', 'frac_mito', 'sample_id', 'gsm_id', 'group_id'

=== Phase 4: Constructing the Final MuData Object ===


... storing 'highly_variable' as categorical


Writing fully integrated multi-omic package to: /home/hwjeong/project-MD/data/multiome.h5mu


... storing 'fragment_file' as categorical



=== SUCCESS: INPUT PREPROCESSING COMPLETE ===
MuData object with n_obs × n_vars = 45755 × 153832
  var:	'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
  2 modalities
    rna:	45755 x 29713
      obs:	'sample_id', 'group_id', 'gsm_id', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'celltype'
      var:	'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
      uns:	'celltype_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
      obsm:	'X_pca', 'X_umap'
      varm:	'PCs'
      obsp:	'connectivities', 'distances'
    atac:	39684 x 124119
      obs:	'n_fragment', 'frac_dup', 'frac_mito', 'sample_id', 'gsm_id', 'group_id', 'fragment_file'
      uns:	'fragment_registry', 'files'
